# 01 Activation Window H01

This notebook tracks the first manual DS experiment after global M5 EDA.

The idea was simple: if a SKU-store has no available sell price yet, those early zero-sales rows may represent pre-activation/ranging rather than true sellable demand. The first test checks whether excluding those rows from challenger training/calibration improves the active holdout forecast and the replenishment tradeoff.

There are two runs here:

- H01a: first quick screen, invalid for this hypothesis because the row cap removed the pre-activation period.
- H01b: full-history run for the selected 60 series, valid for testing the activation-window treatment.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "manual_ds":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
REPORT_DIR = PROJECT_ROOT / "backend" / "reports" / "experiments"

H01A_PATH = REPORT_DIR / "095fa73d-2561-4202-bfb9-fb74490c6c23.report.json"
H01B_PATH = REPORT_DIR / "22aedb84-a49b-499f-a6a4-f0b071c312fd.report.json"

def load_report(path: Path) -> dict:
    with path.open() as f:
        return json.load(f)

h01a = load_report(H01A_PATH)
h01b = load_report(H01B_PATH)


## Run Inventory

Before judging the model result, I want to make sure the experiment actually exercised the treatment. For this hypothesis, the activation counts matter as much as the headline WAPE.

In [ ]:
def run_summary(report: dict) -> dict:
    dataset = report["dataset"]
    activation = report.get("activation_policy") or {}
    return {
        "experiment": report["experiment"]["experiment_name"],
        "rows_used": dataset.get("rows_used"),
        "series_used": dataset.get("series_used"),
        "train_start": dataset.get("train_start"),
        "train_end": dataset.get("train_end"),
        "holdout_start": dataset.get("holdout_start"),
        "holdout_end": dataset.get("holdout_end"),
        "training_rows_excluded": activation.get("training_rows_excluded"),
        "affected_series": activation.get("affected_series"),
        "holdout_pre_activation_rows": activation.get("holdout_pre_activation_rows"),
        "decision": report["promotion_comparison"].get("reason"),
    }

pd.DataFrame([run_summary(h01a), run_summary(h01b)])


## H01a Read

The first run is not interpretable for the activation hypothesis. `training_rows_excluded = 0` and `affected_series = 0`, which means the selected data window had already dropped the early pre-activation period. This is a useful process failure: for a data-window hypothesis, the run configuration can invalidate the test even if the backend spec is correct.

In [ ]:
run_summary(h01a)


## H01b Champion vs Challenger

The full-history run is the valid test. It excluded 9,856 pre-activation training rows across 32 affected series. Now the comparison is fair: same selected series, same date range, same holdout, same active-holdout evaluation slice; only the activation training policy differs.

In [ ]:
metric_keys = [
    "mae",
    "wape",
    "mase",
    "bias_pct",
    "coverage",
    "overstock_dollars",
    "lost_sales_qty",
    "opportunity_cost_stockout",
    "opportunity_cost_overstock",
]

def metric_comparison(report: dict) -> pd.DataFrame:
    baseline = report["baseline"]["holdout_metrics"]
    challenger = report["challenger"]["holdout_metrics"]
    rows = []
    for metric in metric_keys:
        b = baseline.get(metric)
        c = challenger.get(metric)
        rows.append({
            "metric": metric,
            "champion": b,
            "challenger": c,
            "abs_delta": None if b is None or c is None else c - b,
            "pct_delta": None if not b or c is None else (c / b - 1) * 100,
        })
    return pd.DataFrame(rows)

metric_comparison(h01b)


## Business Read

The challenger reduced lost-sales pressure, but the overall tradeoff got worse. WAPE, MASE, bias, overstock dollars, and combined replenishment cost moved the wrong direction. In retail, a more aggressive forecast is not automatically better; it has to clear the service-level / working-capital tradeoff.

In [ ]:
decision_keys = [
    "stockout_days",
    "lost_sales_proxy",
    "lost_sales_units",
    "overstock_dollars",
    "service_level",
    "po_count",
    "combined_cost_proxy",
]

def decision_replay_comparison(report: dict) -> pd.DataFrame:
    champion = report["decision_replay"]["results"]["champion"]
    challenger = report["decision_replay"]["results"]["challenger"]
    rows = []
    for metric in decision_keys:
        b = champion.get(metric)
        c = challenger.get(metric)
        rows.append({
            "metric": metric,
            "champion": b,
            "challenger": c,
            "abs_delta": None if b is None or c is None else c - b,
            "pct_delta": None if not b or c is None else (c / b - 1) * 100,
        })
    return pd.DataFrame(rows)

decision_replay_comparison(h01b)


## Segment Read

The aggregate result rejects the global policy, but the segment breakdown is not a dead end. The treatment helped slower and medium-velocity groups while hurting fast/high-volume series. That suggests the next test should be segment-gated, not another global activation filter.

In [ ]:
def segment_comparison(report: dict) -> pd.DataFrame:
    baseline = report["baseline"].get("segment_metrics") or {}
    challenger = report["challenger"].get("segment_metrics") or {}
    rows = []
    for segment in sorted(set(baseline) | set(challenger)):
        b_payload = baseline.get(segment) or {}
        c_payload = challenger.get(segment) or {}
        b_metrics = b_payload.get("metrics") or {}
        c_metrics = c_payload.get("metrics") or {}
        b_wape = b_metrics.get("wape")
        c_wape = c_metrics.get("wape")
        rows.append({
            "segment": segment,
            "rows": c_payload.get("sample_rows") or b_payload.get("sample_rows"),
            "champion_wape": b_wape,
            "challenger_wape": c_wape,
            "wape_delta": None if b_wape is None or c_wape is None else c_wape - b_wape,
            "champion_bias": b_metrics.get("bias_pct"),
            "challenger_bias": c_metrics.get("bias_pct"),
            "champion_overstock_rate": b_metrics.get("overstock_rate"),
            "challenger_overstock_rate": c_metrics.get("overstock_rate"),
        })
    return pd.DataFrame(rows)

segment_comparison(h01b)


In [ ]:
slice_rows = []
for name, payload in (h01b.get("evaluation_slices") or {}).items():
    b = payload.get("baseline") or {}
    c = payload.get("challenger") or {}
    slice_rows.append({
        "slice": name,
        "rows": payload.get("rows"),
        "primary": payload.get("primary"),
        "guardrail": payload.get("guardrail"),
        "champion_wape": b.get("wape"),
        "challenger_wape": c.get("wape"),
        "wape_delta": None if b.get("wape") is None or c.get("wape") is None else c.get("wape") - b.get("wape"),
        "champion_bias": b.get("bias_pct"),
        "challenger_bias": c.get("bias_pct"),
    })

pd.DataFrame(slice_rows)


## Decision

H01 is rejected as a global policy.

The treatment did not become a champion because it worsened aggregate forecast error, bias, overstock dollars, and combined cost. The useful learning is narrower: activation-aware treatment looks more promising for slow, medium, and late-activation series than for fast/high-volume series.

Next hypothesis:

> Apply activation-aware training only to late-activation and slower/intermittent SKU-store series, while keeping canonical training for fast/high-volume series.

That becomes H02.

## Later Update: Activation Track Decision

H01 was the correct first activation test, but H02 and H03 changed how I read the path.

The signal was repeatable: slow and medium segments improved under activation-aware treatment across H01, H02, and H03. The issue is that the inventory decision tradeoff never cleared. H01 and H02 reduced lost-sales pressure but increased overstock/bias. H03 fixed overstock/bias through routing, but increased stockout and lost-sales risk.

Senior DS decision: stop activation-window mechanics as the primary hypothesis family. Keep the activation insight as a documented segment signal, but move the manual DS loop back to EDA for a new demand driver.
